In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('/content/sample_data/train.txt',sep = ';',header = None,names = ['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

,0
text,0
emotion,0


In [5]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [6]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [7]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [8]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [9]:
df['text'] = df['text'].apply(remove_punc)

In [10]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [11]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [13]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
stop_words = set(stopwords.words('english'))

In [15]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [16]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)
df['text'] = df['text'].apply(remove)

In [17]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


# Different NLP Feature Extraction / Vectorization Methods

In [31]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

documents = [
    "I love machine learning",
    "I love deep learning"
]

# One-Hot Encoding
words = [["I"], ["love"], ["pizza"], ["pasta"]]
encoder = OneHotEncoder(sparse_output=False)
X = encoder.fit_transform(words)

vectorizer = CountVectorizer() # BoW
vectorizer = CountVectorizer(binary=True) # BoW + Binary
vectorizer = CountVectorizer(ngram_range=(1, 3)) # BoW + N-Grams

vectorizer = TfidfVectorizer() # TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1, 2)) # TF-IDF + N-Grams

X = vectorizer.fit_transform(documents)

print(vectorizer.get_feature_names_out())
print(X.toarray())

['deep' 'deep learning' 'learning' 'love' 'love deep' 'love deep learning'
 'love machine' 'love machine learning' 'machine' 'machine learning']
[[0 0 1 1 0 0 1 1 1 1]
 [1 1 1 1 1 1 0 0 0 0]]


In [33]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['text'],df['emotion'],test_size = 0.2,random_state = 42)


#### Use Both TF-DIF & BoW

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
TfIdf = TfidfVectorizer()
BoW = CountVectorizer();


In [35]:
X_train_tfdif = TfIdf.fit_transform(X_train)
X_test_tfdif = TfIdf.transform(X_test)

In [38]:
X_train_tfdif.shape

(12800, 13361)

In [36]:
X_train_BoW = BoW.fit_transform(X_train)
X_test_BoW = BoW.transform(X_test)

In [39]:
X_train_BoW.shape

(12800, 13361)

### Naive Bayes (MultinomialNB) Model

In [41]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

nb = MultinomialNB()

In [45]:
nb.fit(X_train_BoW,y_train)
y_pred = nb.predict(X_test_BoW)

print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.768125
[[899   8   3   0   4  32]
 [ 87 271   0   0  13  56]
 [ 51   6  79   0   4 156]
 [ 40   0   1   6  17  49]
 [ 87  15   0   0 226  69]
 [ 36   4   2   0   2 977]]
              precision    recall  f1-score   support

           0       0.75      0.95      0.84       946
           1       0.89      0.63      0.74       427
           2       0.93      0.27      0.41       296
           3       1.00      0.05      0.10       113
           4       0.85      0.57      0.68       397
           5       0.73      0.96      0.83      1021

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200



### Logistic Regression Model

In [46]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()

In [47]:
lr.fit(X_train_BoW,y_train)
y_pred = lr.predict(X_test_BoW)

print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.8896875
[[879  24   7   1  13  22]
 [ 24 366   0   2  15  20]
 [  5   2 225   1   4  59]
 [  5   0   1  82  19   6]
 [ 19  16   2   7 332  21]
 [ 16   5  31   2   4 963]]
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       946
           1       0.89      0.86      0.87       427
           2       0.85      0.76      0.80       296
           3       0.86      0.73      0.79       113
           4       0.86      0.84      0.85       397
           5       0.88      0.94      0.91      1021

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200

